<a href="https://colab.research.google.com/github/gatai-ok/PO/blob/main/%E0%B9%81%E0%B8%A2%E0%B8%81PO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd
import io
from google.colab import files

def final_perfect_match():
    print("-> อัปโหลดไฟล์ POReport (CSV):")
    po_upload = files.upload()
    po_name = list(po_upload.keys())[0]

    print("-> อัปโหลดไฟล์ รายชื่อสินค้า (Excel .xlsx เท่านั้น!):")
    prod_upload = files.upload()
    prod_name = list(prod_upload.keys())[0]

    # 1. อ่าน PO
    po_df = pd.read_csv(io.BytesIO(po_upload[po_name]))
    po_df = po_df[['SKU_CODE', 'POWBUYQTY']].rename(columns={'SKU_CODE': 'SKU', 'POWBUYQTY': 'QTY'})
    po_df['SKU'] = po_df['SKU'].astype(str).str.split('.').str[0].str.strip()
    po_df = po_df.groupby('SKU')['QTY'].sum().reset_index()

    # 2. อ่านไฟล์รายชื่อสินค้าด้วย read_excel
    # เนื่องจากเป็น Excel ผมจะให้โปรแกรมลองหาหัวตารางเองโดยการวนอ่าน sheet
    raw_data = io.BytesIO(prod_upload[prod_name])

    # ลองอ่านไฟล์ Excel โดยให้ header เป็น None ไปก่อน
    df_raw = pd.read_excel(raw_data, header=None)

    # ค้นหาแถวที่มีคำว่า SKU หรือ รหัสสินค้า
    target_row = -1
    for i in range(min(20, len(df_raw))):
        row_values = df_raw.iloc[i].astype(str).values
        if any('SKU' in str(v).upper() for v in row_values):
            target_row = i
            break

    if target_row != -1:
        # กำหนด header ใหม่ตามแถวที่เจอ
        df_raw.columns = df_raw.iloc[target_row]
        prod_df = df_raw[target_row+1:].copy()

        # คลีนชื่อคอลัมน์
        prod_df.columns = prod_df.columns.astype(str).str.strip()

        # เลือกคอลัมน์
        cols = prod_df.columns.astype(str)
        sku_c = [c for c in cols if 'SKU' in c.upper()][0]
        name_c = [c for c in cols if 'ชื่อ' in c][0]
        dept_c = [c for c in cols if 'แผนก' in c][0]

        prod_df = prod_df[[sku_c, name_c, dept_c]].copy()
        prod_df.columns = ['SKU', 'รายการสินค้า', 'แผนก']
        prod_df['SKU'] = prod_df['SKU'].astype(str).str.split('.').str[0].str.strip()

        # 3. รวมข้อมูล
        res = pd.merge(po_df, prod_df, on='SKU', how='left')
        res = res.fillna('')

        # 4. ดาวน์โหลด
        res.to_excel('รายงานยอดสั่งผลิต.xlsx', index=False)
        files.download('รายงานยอดสั่งผลิต.xlsx')
        print("\n--- สำเร็จ! ดาวน์โหลดไฟล์แล้ว ---")
    else:
        print("\n!!! ไม่พบคำว่า SKU ในไฟล์ Excel เลย กรุณาตรวจสอบว่าอัปโหลดไฟล์ผิดอันหรือไม่ !!!")

final_perfect_match()

-> อัปโหลดไฟล์ POReport (CSV):


Saving POReport01v3 (19).csv to POReport01v3 (19) (24).csv
-> อัปโหลดไฟล์ รายชื่อสินค้า (Excel .xlsx เท่านั้น!):


Saving รายชื่อ OKKN 17-12-24.xlsx to รายชื่อ OKKN 17-12-24 (22).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- สำเร็จ! ดาวน์โหลดไฟล์แล้ว ---
